## Setup

Run the following 2 cells to import all necessary files, libraries and helpers for this exercise.

In [ ]:
!wget -q "https://github.com/PSAM-5020-2026F-A/5020-utils/raw/main/src/audio_utils.py" -O audio_utils.py
!wget -q "https://github.com/PSAM-5020-2026F-A/5020-utils/raw/main/src/data_utils.py"  -O data_utils.py

!mkdir -p data/audio
!wget -q https://github.com/PSAM-5020-2026F-A/5020-utils/raw/refs/heads/main/datasets/audio/hood-internet.tar.gz
!tar -xzf ./hood-internet.tar.gz -C ./data/audio/ && rm -rf hood-internet.tar.gz*

In [ ]:
import librosa
import matplotlib.pyplot as plt
import pandas as pd
import random

from IPython.display import Audio
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer
from os import listdir

## Shazam !

Yep. Let's replicate the Shazam algorithm. It's the same type of classification task we've been looking at, but instead of classifying a piece of audio into one of $3$ classes, the algorithm has to pick which of the millions of songs the piece of audio most likely came from.

We've created some simple classification models by manually extracting features that we think are important for the differentiation of our classes. This is ok if we're only dealing with a few classes, but it doesn't scale. If we had to classify audio into hundreds, or thousands, of classes, looking at graphs of relative frequencies and loudness for all classes becomes intractable.

We need something a little more systematic. Something that automatically extracts meaningful numeric information from audio, independent of the type of audio.

Let's look at the frequency domain again, and use the librosa library to help us extract consistent frequency information from audio files.

## Librosa Feature Extraction

Librosa is useful because it has many feature extraction functions to actually get meaningful frequency information from our audios files:

[Librosa Reference and Documentation](https://librosa.org/doc/latest/feature.html)

### Some example functions

- `librosa.feature.spectral_centroid` (extracts average frequency)
- `librosa.feature.spectral_bandwidth` (extracts frequency range)
- `librosa.feature.spectral_rolloff(roll_percent=0.01)` (extarcts smallest frequency)
- `librosa.feature.spectral_rolloff(roll_percent=0.99)` (extracts largest frequency)

Let's load a song and see what running these looks like:

### ❗️❗️ WARNING ❗️❗️

The songs we'll be using are from mixtapes by [The Hood Internet](https://www.thehoodinternet.com/). They were part of a wave of _mashup_ producers in the early $2\text{,}000\text{s}$ that mixed _classic_ american indie, pop and rap music from other time periods.

We're using their mixtapes here mostly because: they're available freely on the internet, they encompass a good variety of songs and styles, and there are a good number of songs; not too few nor too many.

The lyrics mostly come from the rap songs being sampled and contain references to all the things that rap artists were rapping about in the $90\text{s}$ and $00\text{s}$ (sex, drugs, guns, money...).

We definitely don't have to listen to the songs to complete the exercise.

We can consider alternative datasets for this exercise. Feel free to adapt the exercise to use files from different albums, just indicate that here in the notebook.

In [ ]:
# First, let's create a list with all audio file names from our album directory

TRACKS_DIR = "./data/audio/hood-internet/mixtape04"
fnames = sorted([f for f in listdir(TRACKS_DIR) if f.endswith("mp3")])

# And take a listen to a random one

print(fnames[5])
y,sr = librosa.load(fnames[5])

plt.plot(y)
plt.show()

display(Audio(y, rate=sr))

### Librosa Features Extraction

We'll leverage some of the builtin functionality of `librosa` to extract frequency information about our songs without having to do too much manual feature extraction.

The idea is to create a unique _fingerprint_ for each song based on the different frequencies present at different times.

The code below extracts the center, lowest and highest frequencies of our signal, throughout the song.

Because these functions were written to work with multi-channel audio, they'll always return a list of lists. In our case, we just have a single channel, so our result will be at index $0$ of the returned list.

That's why we have `sc[0]`, `smin[0]`, `smax[0]` below.

In [ ]:
# average frequency
sc = librosa.feature.spectral_centroid(y=y)

# min and max frequencies
smin = librosa.feature.spectral_rolloff(y=y, roll_percent=0.01)
smax = librosa.feature.spectral_rolloff(y=y, roll_percent=0.99)

plt.plot(sc[0])
plt.plot(smin[0])
plt.plot(smax[0])
plt.show()

This is a start.

The functions give us multiple values because the avg/min/max frequency changes over time, but we can average the `spectral_centroid` and the two `spectral_rolloff` frequencies to get $3$ values that represent our entire audio file.

In [ ]:
sc_val = sum(sc[0]) / len(sc[0])
smin_val = sum(smin[0]) / len(smin[0])
smax_val = sum(smax[0]) / len(smax[0])

plt.plot([0, len(sc[0])], [sc_val, sc_val])
plt.plot([0, len(smin[0])], [smin_val, smin_val])
plt.plot([0, len(smax[0])], [smax_val, smax_val])
plt.show()

Not bad.

### MFCCs

But, we can extend this idea with Mel-Frequency Cepstral Coefficients (MFCC). MFCCs are a compact, numerical representation of an audio signal's frequency characteristics, that divide an audio frequency range into bands based on human hearing sensitivities. So, instead of just $3$ frequencies, like we got above, the total frequency range is divided into $10$ to $20$ values that represent our audio signal.

And of course `librosa` has an `mfcc()` function:

In [ ]:
mfccs = librosa.feature.mfcc(y=y, n_mfcc=10)
print(mfccs.shape)

for vs in mfccs:
  plt.plot(vs)
plt.show()

### Average MFCCs

Like the `spectral_centroid()` function, the `mfcc()` function gives us multiple values for each frequency channel/band, since these change over time as the audio changes. We'll assume that one value per frequency band is enough and average them over time, so we end up with only $10$ numbers to represent our audio.

In [ ]:
mfccs_avg = mfccs.mean(axis=1)
print(mfccs_avg.shape)

for vs in mfccs_avg:
  plt.plot([0, len(mfccs[0])], [vs, vs])
plt.show()

### Back to Shazam

This is what's behind the [Shazam algorithm](https://www.youtube.com/watch?v=b6xeOLjeKs0): a large database of songs, where each song is described with only a few features so they can be organized, searched, queried and compared very very quickly.

Another [video](https://www.youtube.com/watch?v=kMNSAhsyiDg) reference.

Let's implement something like this.

We'll focus on a single album, with about $30$ songs, but the same strategy (with minor modifications) will work for datasets of many more songs.

Here's the outline of what we have to do:

1. Extract MFCC features from all songs in our database (single album)
2. Create a dataset of song features
3. Given an input sound clip: extract its MFCC features
4. Find closest song in database by feature distance

#### Helper Function

Let's define a function that, given a list of samples, extracts $10$ to $20$ average MFCC features.

It's the steps above, but wrapped in a function for easier re-use.

In [ ]:
# Helper function:
#   given a list of samples, 
#   returns a list of extracted mfcc values

def get_features(y, n_mfcc=20):
  mfccs = librosa.feature.mfcc(y=y, n_mfcc=n_mfcc).mean(axis=1)
  mfeats = [float(fcc) for fcc in mfccs]
  return mfeats

### Song Features

We have a list of song names. We have a function to extract features.

Let's create a dataset of features for all songs.

In [ ]:
# TODO: create list to hold features
# TODO: iterate over song filenames
#       open song, get samples
#       get features
#       add song to dataset

### Test clip

Let's pick a 5-second clip from a random song and also extract its MFCC features.

Let's extract features from both the middle of the song and a random 5-second clip.

In [ ]:
# Track idx
sidx = 1
fpath = f"{TRACKS_DIR}/{fnames[sidx]}"
y_in, sr_in = librosa.load(fpath)

# get 5 seconds from middle of song
midx = int(len(y_in) // 2)
half_win = int(2.5 * sr_in)
samples_mid_5sec = y_in[midx - half_win : midx + half_win]

# get random 5-sec clip
clip_len_samples = sr * 5

# select random start location
clip_start = random.randrange(0, len(y) - clip_len_samples)

# samples start at random location and end at start + 5 seconds
samples_random_5sec = y_in[clip_start : clip_start + clip_len_samples]

feat_in_mid = get_features(samples_mid_5sec)
feat_in_random = get_features(samples_random_5sec)

### The Algorithm

We have a clip we want to find in our dataset.

We have features for everything.

We just have to iterate over our dataset and find the song with the shortest distance to our clip.

It helps to create a function to compare two lists of features, and return the Euclidean distance between them.

In [ ]:
# TODO: write a function that returns the distance between two lists of MFCC features.
#       Both lists have the same length.
#       We want to square the difference between corresponding features
#       Sum those differences, and take the square root
#       The zip() function might help


def mfcc_distance(mfccA, mfccB):
  diffs = []
  # TODO: loop over features
  sum = 0
  dist = 0
  return dist

### Apply the function to the data.

Now we iterate over the songs in the dataset and calculate their distance from the clip.

In [ ]:
# TODO: iterate over the songs

### Find index of shortest distance

We don't want the distance, but the index of the song that gave the shortest distance.

This is called the `argmin` of the list.

In [ ]:
# TODO: implement argmin